# HFCS analysis dataset

Harmonised household-level data for cross-country wealth-by-age profiles. The original Parquet exports remain unchanged. Each observation corresponds to one household, survey wave and imputation.

Main samples: all households and households with no reported inheritance. Inheritance status is inferred from the transfer module and acquisition of the household main residence. Uncertain cases are retained in the master data but excluded from the no-inheritance sample.

This is a repeated-cross-sectional dataset. Household identifiers are not treated as longitudinal links.

## Settings

In [1]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

WAVES = (2010, 2014, 2017, 2021, 2023)
IMPUTATIONS = (1, 2, 3, 4, 5)
PRICE_YEAR = 2023
MIN_QUARTILE_N = 40
MIN_AGE = 20
MAX_AGE = 110

REPO_ROOT_OVERRIDE = None
INPUT_ROOT_OVERRIDE = None
CPI_CSV_OVERRIDE = None  # optional CSV with country,year,hicp_index


def find_repo():
    setting = REPO_ROOT_OVERRIDE or os.environ.get("HFCS_REPO_ROOT")
    if setting:
        return Path(setting).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        if (parent / "hfcs_file_inventory.txt").exists() or (
            (parent / "HFCS").is_dir() and (parent / "load_hfcs.ipynb").is_file()
        ):
            return parent
    raise FileNotFoundError("Run the notebook from Research-Assistant-Git, or set REPO_ROOT_OVERRIDE")


REPO = find_repo()
INPUT = (Path(INPUT_ROOT_OVERRIDE).expanduser().resolve() if INPUT_ROOT_OVERRIDE else
         REPO.parent / "HFCS Data Download" / "HFCS_selected_raw")
OUTPUT = INPUT / "analysis_ready"
OUTPUT.mkdir(parents=True, exist_ok=True)

paths = [INPUT / f"hfcs_{year}_{table}_imp{imp}.parquet"
         for year in WAVES for imp in IMPUTATIONS for table in ("h", "d", "p")]
missing_paths = [str(p) for p in paths if not p.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Missing {len(missing_paths)} Parquet exports, e.g. {missing_paths[:3]}")

print("Input:", INPUT)
print("Output:", OUTPUT)
print("Parquet exports:", len(paths))

Input: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download\HFCS_selected_raw
Output: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download\HFCS_selected_raw\analysis_ready
Parquet exports: 75


## Price indices

Eurostat annual all-items HICP (`prc_hicp_ainr`, ECOICOP v2, 2025=100). Wealth is expressed in 2023 euros. The price year is approximated by the HFCS wave year because interview dates differ by country.

The official index is downloaded once and cached locally. Alternatively provide `CPI_CSV_OVERRIDE` with the columns `country`, `year`, `hicp_index`. Source: https://ec.europa.eu/eurostat/databrowser/view/prc_hicp_ainr/default/table?lang=en

In [ ]:
HICP_CACHE = INPUT / "hicp_annual_eurostat.csv"
HICP_URL = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/prc_hicp_ainr"


def eurostat_json_to_frame(data):
    dims = data["id"]
    sizes = list(map(int, data["size"]))
    levels = []
    for name, size in zip(dims, sizes):
        lookup = data["dimension"][name]["category"]["index"]
        ordered = [None] * size
        for label, idx in lookup.items():
            ordered[int(idx)] = str(label)
        levels.append(ordered)
    records = []
    for flat, value in data.get("value", {}).items():
        remainder = int(flat)
        positions = [0] * len(sizes)
        for k in range(len(sizes) - 1, -1, -1):
            positions[k] = remainder % sizes[k]
            remainder //= sizes[k]
        row = {name: levels[k][idx] for k, (name, idx) in
               enumerate(zip(dims, positions))}
        row["hicp_index"] = float(value)
        records.append(row)
    return pd.DataFrame(records)


def load_hicp():
    if CPI_CSV_OVERRIDE:
        path = Path(CPI_CSV_OVERRIDE).expanduser().resolve()
        frame = pd.read_csv(path)
    elif HICP_CACHE.is_file():
        frame = pd.read_csv(HICP_CACHE)
    else:
        query = {"freq": "A", "unit": "I25", "coicop18": "TOTAL",
                 "sinceTimePeriod": "2008", "untilTimePeriod": str(PRICE_YEAR)}
        url = HICP_URL + "?" + urlencode(query)
        try:
            req = Request(url, headers={"User-Agent": "HFCS-research/1.0"})
            with urlopen(req, timeout=60) as response:
                raw = json.load(response)
        except Exception as exc:
            raise RuntimeError(
                "Cannot load Eurostat HICP. Download a country-year index CSV "
                "and set CPI_CSV_OVERRIDE, or check the internet connection."
            ) from exc
        frame = eurostat_json_to_frame(raw)
        if frame.empty:
            raise RuntimeError("Eurostat returned no HICP observations")
        frame = frame.rename(columns={"geo": "country", "time": "year"})
        frame = frame[["country", "year", "hicp_index"]]
        frame.to_csv(HICP_CACHE, index=False)
    required = {"country", "year", "hicp_index"}
    if not required.issubset(frame):
        raise ValueError(f"HICP input requires {sorted(required)}")
    frame = frame[list(required)].copy()
    frame["country"] = frame["country"].astype("string").str.upper().replace({"EL": "GR"})
    frame["year"] = pd.to_numeric(frame["year"], errors="raise").astype("int32")
    frame["hicp_index"] = pd.to_numeric(frame["hicp_index"], errors="coerce")
    frame = frame.loc[frame.hicp_index.gt(0)]
    conflicts = frame.groupby(["country", "year"])["hicp_index"].nunique()
    if conflicts.gt(1).any():
        raise AssertionError("Conflicting HICP country-year indices")
    frame = frame.drop_duplicates(["country", "year"])
    return frame


hicp = load_hicp()
reference_prices = hicp.loc[hicp.year.eq(PRICE_YEAR), ["country", "hicp_index"]].rename(
    columns={"hicp_index": "hicp_reference"})
print("HICP country-years:", len(hicp), "| reference year:", PRICE_YEAR)
display(hicp.head())

## Variable selection and coding

In [ ]:
KEY = ["sa0100", "sa0010", "im0100"]
H_COLUMNS = ["hw0010", "hw0010a", "hh0100", "hh0110", "hb0600", "hh0201", "hh0401"]
D_COLUMNS = ["dn3001", "da3001", "da1000", "da1110", "da1110i",
             "dl1000", "di2000", "dhageh1", "dhageh1b"]


def available_columns(path, wanted):
    actual = {x.lower(): x for x in pq.ParquetFile(path).schema_arrow.names}
    return [actual[name] for name in wanted if name in actual]


# The number of recorded transfers differs between waves.
type_columns = sorted({x.lower() for year in WAVES for imp in IMPUTATIONS
                       for x in pq.ParquetFile(INPUT / f"hfcs_{year}_h_imp{imp}.parquet")
                       .schema_arrow.names if re.fullmatch(r"hh050\d+", x, flags=re.I)})
if not type_columns:
    raise ValueError("No HH050x variables found")
print("Transfer type variables:", type_columns)


def numeric(x):
    return pd.to_numeric(x, errors="coerce").astype("float64")


def age_to_lower(age):
    # Reference-person ages are grouped according to the published HFCS bands.
    boundaries = np.array([0, 7, 14, 16, 20, 25, 30, 35, 40, 45, 50,
                           55, 60, 65, 70, 75, 80, 85], dtype=float)
    values = numeric(age).to_numpy(dtype=float)
    idx = np.searchsorted(boundaries, values, side="right") - 1
    out = np.full(len(values), np.nan)
    ok = np.isfinite(values) & (values >= 0) & (values <= MAX_AGE)
    out[ok] = boundaries[idx[ok]]
    return pd.Series(out, index=age.index, dtype="float64")


BRACKET_LOWER = [0, 7, 14, 16, 20, 25, 30, 35, 40, 45,
                 50, 55, 60, 65, 70, 75, 80, 85]
BRACKET_ORDINAL = dict(zip(range(1, 19), BRACKET_LOWER))


def decode_derived_age_brackets(raw, wave):
    values = numeric(raw)
    present = set(values.dropna().astype(int).tolist())
    ordinal = set(BRACKET_ORDINAL)
    lower = set(BRACKET_LOWER)
    if not present:
        return values, "missing"
    # 2010 contains both ordinal and lower-bound representations in some exports.
    if present.issubset(ordinal) and not present.issubset(lower):
        return values.map(BRACKET_ORDINAL).astype("float64"), "ordinal"
    if present.issubset(lower) and not present.issubset(ordinal):
        return values, "lower_bound"
    if present.issubset(lower) and present.issubset(ordinal):
        raise ValueError(f"Ambiguous age-bracket coding in {wave}: {sorted(present)}")
    if present.issubset(lower):
        return values, "lower_bound"
    raise ValueError(f"Unrecognised DHAGEH1B values in {wave}: {sorted(present - lower - ordinal)}")


def readable_age_bin(lower):
    if pd.isna(lower):
        return pd.NA
    x = int(lower)
    if x == 85:
        return "85+"
    if x in (0, 7, 14):
        return {0: "0-6", 7: "7-13", 14: "14-15"}[x]
    return f"{x}-{19 if x == 16 else x+4}"


def read_h_d(wave, imp):
    hpath = INPUT / f"hfcs_{wave}_h_imp{imp}.parquet"
    dpath = INPUT / f"hfcs_{wave}_d_imp{imp}.parquet"
    hcols = KEY + H_COLUMNS + type_columns
    dcols = KEY + D_COLUMNS
    h = pd.read_parquet(hpath, columns=available_columns(hpath, hcols))
    d = pd.read_parquet(dpath, columns=available_columns(dpath, dcols))
    h.columns = h.columns.str.lower()
    d.columns = d.columns.str.lower()
    for k in KEY:
        if k not in h or k not in d:
            raise KeyError(f"Missing {k} in {wave}, imputation {imp}")
    # Only the household-level derived age is needed; P is not merged into H.
    for col in H_COLUMNS + type_columns:
        if col not in h:
            h[col] = np.nan
    for col in D_COLUMNS:
        if col not in d:
            d[col] = np.nan
    for table in (h, d):
        table["sa0100"] = table["sa0100"].astype("string").str.strip().str.upper()
        table["sa0010"] = table["sa0010"].astype("string").str.strip()
        table["im0100"] = numeric(table["im0100"])
    if h.duplicated(KEY).any() or d.duplicated(KEY).any():
        raise AssertionError(f"Duplicate H/D key in wave {wave}, imputation {imp}")
    if not h.im0100.eq(imp).all() or not d.im0100.eq(imp).all():
        raise AssertionError(f"Incorrect imputation number: {wave}, {imp}")
    merged = h.merge(d, on=KEY, how="outer", validate="one_to_one", indicator=True)
    if not merged._merge.eq("both").all():
        raise AssertionError(f"Household and derived keys do not match: {wave}, {imp}")
    return merged.drop(columns="_merge")

## Inheritance status

The transfer module distinguishes gifts (`HH050x = 1`) from inheritances (`HH050x = 2`). Inherited main residences (`HB0600 = 3`) are counted as inheritances even if `HH0100` does not report one. Code `5` for mixed acquisition is ambiguous, as is `6` in the historical Spanish coding.

A confirmed non-inheritor must either report no transfers or have a complete list consisting only of gifts, and their main-residence acquisition must not indicate an inheritance. Missing details are never interpreted as a negative response.

In [ ]:
def prepare_chunk(wave, imp, cpi_lookup):
    df = read_h_d(wave, imp)
    n = len(df)
    country_raw = df.sa0100.copy()
    country = country_raw.replace({"E1": "ES"})
    df["country_raw"] = country_raw
    df["country"] = country
    df["wave"] = np.int16(wave)
    df["imputation"] = np.int8(imp)
    df["household_id"] = df.sa0010.astype("string")
    df["wave_id"] = country_raw.astype("string") + "_" + str(wave)
    # E1 is an additional Spanish vintage. Do not assume the survey reference year.
    df["price_year"] = np.where(country_raw.eq("E1"), np.nan, float(wave))
    df["reference_year_verified"] = ~country_raw.eq("E1")

    df["net_wealth"] = numeric(df.dn3001)  # Negative wealth is valid.
    df["gross_assets"] = numeric(df.da3001)
    df["real_assets"] = numeric(df.da1000)
    df["main_residence_value"] = numeric(df.da1110)
    df["total_debt"] = numeric(df.dl1000)
    df["gross_income"] = numeric(df.di2000)
    w = numeric(df.hw0010)
    w_it = numeric(df.hw0010a)
    it_hist_weight = country.eq("IT") & w_it.gt(0)
    df["hh_weight"] = w.where(~it_hist_weight, w_it).where(lambda z: z.gt(0))
    df["weight_source"] = np.where(it_hist_weight, "hw0010a", "hw0010")
    df["weight_source"] = pd.Series(df.weight_source, index=df.index, dtype="string")
    df.loc[df.hh_weight.isna(), "weight_source"] = "invalid_or_missing"

    exact = numeric(df.dhageh1)
    exact = exact.where(exact.between(16, MAX_AGE))
    lower = pd.Series(np.nan, index=df.index, dtype="float64")
    schemes = {}
    for c, ids in df.groupby("country_raw", dropna=False).groups.items():
        lower.loc[ids], scheme = decode_derived_age_brackets(df.loc[ids, "dhageh1b"], wave)
        schemes[str(c)] = scheme
    lower_from_exact = age_to_lower(exact)
    age_disagreement = exact.notna() & lower.notna() & ~np.isclose(
        lower, lower_from_exact, equal_nan=True)
    # When both measures conflict, use the exact age to assign the age group.
    df["age_years"] = exact
    df["age_lower"] = lower.where(~age_disagreement, lower_from_exact).fillna(lower_from_exact)
    df["age_group"] = df.age_lower.map(readable_age_bin).astype("string")
    df["age_information"] = pd.Series(np.where(exact.notna(), "exact", np.where(
        lower.notna(), "brackets_only", "missing")), index=df.index, dtype="string")
    df["age_bracket_conflict"] = age_disagreement.astype("bool")
    df["age_bracket_scheme"] = df.country_raw.map(schemes).astype("string")

    receipt = numeric(df.hh0100)
    count = numeric(df.hh0110)
    types = df[type_columns].apply(pd.to_numeric, errors="coerce")
    known_type_count = types.isin([1, 2]).sum(axis=1)
    unexpected_type_codes = (types.notna() & ~types.isin([1, 2, -1, -2])).any(axis=1)
    from_transfer = types.eq(2).any(axis=1)
    gift_reported = types.eq(1).any(axis=1)
    listed_gifts_only = (receipt.eq(1) & count.ge(1) &
                         count.le(len(type_columns)) & known_type_count.ge(count) &
                         ~from_transfer)

    home_acquisition = numeric(df.hb0600)
    home_ownership = numeric(df.da1110i)
    inherited_home = home_acquisition.eq(3)
    # DA1110i = 0 is a non-owner; owners need an identifiable acquisition type.
    home_excludes_inh = home_ownership.eq(0) | home_acquisition.isin([1, 2, 4])
    # 5 can mean a gift OR inheritance; 6 is not classified as a non-inheritance.
    confirmed_inheritance = from_transfer | inherited_home
    confirmed_no_inheritance = ((receipt.eq(2) | listed_gifts_only) &
                                home_excludes_inh & ~confirmed_inheritance &
                                ~unexpected_type_codes)
    df["inheritance_status"] = pd.Series(np.select(
        [confirmed_inheritance, confirmed_no_inheritance],
        ["confirmed_inheritance", "no_reported_inheritance"], default="uncertain"
    ), index=df.index, dtype="string")
    df["reported_gift"] = (gift_reported | home_acquisition.eq(4)).astype("bool")
    df["inherited_main_residence"] = inherited_home.astype("bool")
    df["inheritance_record_conflict"] = (receipt.eq(2) & confirmed_inheritance).astype("bool")
    df["transfer_count_covered"] = (receipt.eq(1) & count.ge(1) &
                                    count.le(len(type_columns)) & known_type_count.ge(count)).astype("bool")
    df["unexpected_transfer_type_code"] = unexpected_type_codes.astype("bool")
    df["home_acquisition_known"] = home_excludes_inh | inherited_home
    df["hh0100"] = receipt
    df["hh0110"] = count
    df["hb0600"] = home_acquisition
    df["da1110i"] = home_ownership
    for name in type_columns:
        df[name] = numeric(df[name])

    country_prices = cpi_lookup.loc[cpi_lookup.year.eq(wave),
                                    ["country", "hicp_index", "hicp_reference"]]
    df = df.merge(country_prices, on="country", how="left", validate="many_to_one")
    df.loc[~df.reference_year_verified, ["hicp_index", "hicp_reference"]] = np.nan
    df["net_wealth_real"] = df.net_wealth * df.hicp_reference / df.hicp_index
    df["gross_assets_real"] = df.gross_assets * df.hicp_reference / df.hicp_index
    df["gross_income_real"] = df.gross_income * df.hicp_reference / df.hicp_index

    df["eligible_all"] = (df.net_wealth_real.notna() & df.age_lower.ge(MIN_AGE) &
                          df.age_lower.le(85) & df.hh_weight.gt(0) &
                          df.reference_year_verified & ~df.age_bracket_conflict)
    df["eligible_no_inheritance"] = (df.eligible_all &
                                      df.inheritance_status.eq("no_reported_inheritance"))
    df["eligible_age_20_65"] = (df.eligible_all & df.age_lower.lt(65))

    keep = (["country", "country_raw", "wave", "wave_id", "price_year", "imputation",
             "household_id", "reference_year_verified", "net_wealth", "net_wealth_real",
             "gross_assets", "gross_assets_real", "real_assets", "main_residence_value",
             "total_debt", "gross_income", "gross_income_real", "age_years", "age_lower",
             "age_group", "age_information", "age_bracket_scheme", "age_bracket_conflict",
             "hh_weight", "weight_source", "hicp_index", "hicp_reference",
             "inheritance_status", "reported_gift", "inherited_main_residence",
             "inheritance_record_conflict", "transfer_count_covered",
             "unexpected_transfer_type_code", "home_acquisition_known",
             "hh0100", "hh0110", "hb0600", "da1110i", "eligible_all",
             "eligible_no_inheritance", "eligible_age_20_65"] + type_columns)
    out = df[keep].copy()
    # Stable dtypes across country-waves for ParquetWriter.
    string_cols = ["country", "country_raw", "wave_id", "household_id", "weight_source",
                   "age_group", "age_information", "age_bracket_scheme", "inheritance_status"]
    for col in string_cols:
        out[col] = out[col].astype("string")
    out["wave"] = out.wave.astype("int16")
    out["imputation"] = out.imputation.astype("int8")
    for col in out.columns.difference(string_cols + ["wave", "imputation"]):
        if out[col].dtype == "object":
            raise TypeError(f"Unexpected object dtype: {col}")
    if len(out) != n:
        raise AssertionError("Unexpected row count after country price merge")
    return out

## Build household dataset

The intermediate file is written sequentially by wave and imputation. Country-wave imputations are not collapsed. The final output stays outside the Git repository.

In [ ]:
hicp_lookup = hicp.merge(reference_prices, on="country", how="left", validate="many_to_one")
STAGE = OUTPUT / "_hfcs_stage.parquet"
FINAL = OUTPUT / "hfcs_analysis.parquet"

# Check that inflation data cover every included country and wave.
all_countries = set()
for wave in WAVES:
    schema = pq.ParquetFile(INPUT / f"hfcs_{wave}_h_imp1.parquet").schema_arrow.names
    k = next((x for x in schema if x.lower() == "sa0100"), None)
    if k is None:
        raise KeyError(f"Missing SA0100 in {wave}")
    observed = pd.read_parquet(INPUT / f"hfcs_{wave}_h_imp1.parquet", columns=[k])[k]
    for country in observed.astype("string").unique():
        all_countries.add((wave, str(country)))
missing_hicp = sorted((wave, c) for wave, c in all_countries if c != "E1" and
                      not ((hicp_lookup.country.eq(c) & hicp_lookup.year.eq(wave) &
                            hicp_lookup.hicp_reference.notna()).any()))
if missing_hicp:
    raise ValueError(f"HICP missing for country-waves: {missing_hicp}")

writer = None
try:
    for wave in WAVES:
        for imp in IMPUTATIONS:
            part = prepare_chunk(wave, imp, hicp_lookup)
            arrow = pa.Table.from_pandas(part, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(STAGE, arrow.schema, compression="zstd")
            else:
                arrow = arrow.cast(writer.schema, safe=False)
            writer.write_table(arrow)
            print(f"{wave}, imputation {imp}: {len(part):,}")
            del part, arrow
finally:
    if writer is not None:
        writer.close()

if writer is None:
    raise AssertionError("No rows written")
print("Intermediate dataset:", STAGE)

## Age-specific wealth quartiles

Groups are calculated separately for each country, imputation and five-year age band, pooling survey waves after conversion to constant prices. This is a contemporaneous wealth ranking, not a fixed household group followed over time.

The quartiles for the no-inheritance sample are calculated *within that sample*. Small or degenerate groups are left unclassified. The five imputations remain separate.

In [ ]:
rank_cols = ["country", "country_raw", "wave", "imputation", "age_lower",
             "net_wealth_real", "hh_weight", "eligible_all", "eligible_no_inheritance"]
rank_input = pd.read_parquet(STAGE, columns=rank_cols)
rank_input = rank_input.loc[rank_input.country_raw.ne("E1")].copy()


def weighted_breaks(value, weight):
    x = np.asarray(value, dtype="float64")
    w = np.asarray(weight, dtype="float64")
    ok = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[ok], w[ok]
    if len(x) < MIN_QUARTILE_N or len(np.unique(x)) < 4:
        return None
    order = np.argsort(x, kind="stable")
    x, w = x[order], w[order]
    p = (np.cumsum(w) - 0.5 * w) / w.sum()
    breaks = np.interp([0.25, 0.50, 0.75], p, x)
    if not (breaks[0] < breaks[1] < breaks[2]):
        return None
    return breaks


def estimate_cutoffs(df, eligibility_col, label):
    rows = []
    data = df.loc[df[eligibility_col]].copy()
    for (country, imp, age), g in data.groupby(["country", "imputation", "age_lower"],
                                                observed=True, sort=False):
        cuts = weighted_breaks(g.net_wealth_real, g.hh_weight)
        rows.append({"country": str(country), "imputation": int(imp), "age_lower": age,
                     "sample": label, "n": len(g),
                     "q25": cuts[0] if cuts is not None else np.nan,
                     "q50": cuts[1] if cuts is not None else np.nan,
                     "q75": cuts[2] if cuts is not None else np.nan})
    return pd.DataFrame(rows)


all_cuts = estimate_cutoffs(rank_input, "eligible_all", "all")
no_inh_cuts = estimate_cutoffs(rank_input, "eligible_no_inheritance", "no_inheritance")
cutoffs = pd.concat([all_cuts, no_inh_cuts], ignore_index=True)
cutoffs.to_csv(OUTPUT / "hfcs_wealth_quartile_cutoffs.csv", index=False)


def attach_quartiles(df, cuts, name):
    cols = ["country", "imputation", "age_lower", "q25", "q50", "q75"]
    b = df.merge(cuts[cols], on=["country", "imputation", "age_lower"],
                 how="left", validate="many_to_one")
    v = b.net_wealth_real.to_numpy(dtype=float)
    a = b[["q25", "q50", "q75"]].to_numpy(dtype=float)
    available = np.isfinite(v) & np.isfinite(a).all(axis=1)
    result = np.zeros(len(df), dtype="int8")
    result[available] = 1 + (v[available, None] > a[available]).sum(axis=1)
    df[name] = result   # 0 means not classified
    return df


writer = None
reader = pq.ParquetFile(STAGE)
try:
    for batch in reader.iter_batches(batch_size=60_000):
        block = batch.to_pandas()
        block = attach_quartiles(block, all_cuts, "wealth_quartile_all")
        block = attach_quartiles(block, no_inh_cuts, "wealth_quartile_no_inheritance")
        block.loc[~block.eligible_all, "wealth_quartile_all"] = 0
        block.loc[~block.eligible_no_inheritance, "wealth_quartile_no_inheritance"] = 0
        arrow = pa.Table.from_pandas(block, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(FINAL, arrow.schema, compression="zstd")
        else:
            arrow = arrow.cast(writer.schema, safe=False)
        writer.write_table(arrow)
finally:
    if writer is not None:
        writer.close()

del rank_input
STAGE.unlink()
print("Analysis data:", FINAL)
print("Cutoffs:", OUTPUT / "hfcs_wealth_quartile_cutoffs.csv")

## Validation and sample audit

All checks below use household observations in each imputation. Summary counts should be interpreted per imputation, not as independent repeated households.

In [ ]:
audit_columns = ["country", "country_raw", "wave", "imputation", "household_id", "net_wealth",
                 "net_wealth_real", "age_years", "age_lower", "age_information",
                 "age_bracket_conflict", "hh_weight", "inheritance_status",
                 "inheritance_record_conflict", "unexpected_transfer_type_code",
                 "eligible_all", "eligible_no_inheritance",
                 "wealth_quartile_all", "wealth_quartile_no_inheritance"]
a = pd.read_parquet(FINAL, columns=audit_columns)
if a.duplicated(["country_raw", "wave", "imputation", "household_id"]).any():
    raise AssertionError("Duplicate household within country, wave and imputation")
assert not a.loc[a.eligible_all, ["net_wealth_real", "age_lower", "hh_weight"]].isna().any().any()
assert a.loc[a.eligible_no_inheritance, "inheritance_status"].eq("no_reported_inheritance").all()
assert not a.loc[a.country_raw.eq("E1"), "eligible_all"].any()
assert a.wealth_quartile_all.between(0, 4).all()
assert a.wealth_quartile_no_inheritance.between(0, 4).all()

country_wave = (
    a.assign(is_inh=a.inheritance_status.eq("confirmed_inheritance"),
             is_no_inh=a.inheritance_status.eq("no_reported_inheritance"),
             is_uncertain=a.inheritance_status.eq("uncertain"))
    .groupby(["wave", "country_raw", "country", "imputation"], as_index=False)
    .agg(households=("net_wealth", "size"),
         net_wealth_observed=("net_wealth", "count"),
         exact_age=("age_years", "count"),
         age_available=("age_lower", "count"),
         age_conflicts=("age_bracket_conflict", "sum"),
         valid_weight=("hh_weight", "count"),
         inheritances=("is_inh", "sum"),
         no_reported_inheritance=("is_no_inh", "sum"),
         uncertain=("is_uncertain", "sum"),
         inheritance_conflicts=("inheritance_record_conflict", "sum"),
         unexpected_type_codes=("unexpected_transfer_type_code", "sum"),
         eligible_all=("eligible_all", "sum"),
         eligible_no_inheritance=("eligible_no_inheritance", "sum"))
)
assert (country_wave[["inheritances", "no_reported_inheritance", "uncertain"]]
        .sum(axis=1).eq(country_wave.households)).all()
country_wave["unknown_pct"] = (100 * country_wave.uncertain / country_wave.households).round(2)
country_wave.to_csv(OUTPUT / "hfcs_build_audit.csv", index=False)

# A compact country-wave overview is also written for planning the LOWESS fits.
summary = country_wave.loc[country_wave.imputation.eq(1), ["wave", "country_raw", "country",
    "households", "exact_age", "age_available", "age_conflicts", "inheritances",
    "no_reported_inheritance", "uncertain", "unknown_pct", "unexpected_type_codes",
    "eligible_all",
    "eligible_no_inheritance"]].copy()
summary["usable_no_inh"] = summary.eligible_no_inheritance.gt(0)
summary.to_csv(OUTPUT / "hfcs_country_wave_samples.csv", index=False)

print("Households in one imputation:", int(a.loc[a.imputation.eq(1)].shape[0]))
print("Country-wave combinations:", len(summary))
print("No-inheritance eligible (imputation 1):", int(summary.eligible_no_inheritance.sum()))
print("Age conflicts (imputation 1):", int(summary.age_conflicts.sum()))
print("Inheritance contradictions (imputation 1):", int(country_wave.loc[
    country_wave.imputation.eq(1), "inheritance_conflicts"].sum()))
print("Unexpected transfer codes (imputation 1):", int(country_wave.loc[
    country_wave.imputation.eq(1), "unexpected_type_codes"].sum()))
display(summary.sort_values(["country", "wave"]).head(25))

## Outputs

- `hfcs_analysis.parquet`: one household-wave-imputation dataset, with nominal and real wealth, age information, survey weights, inheritance status, eligibility flags and wealth quartiles.
- `hfcs_wealth_quartile_cutoffs.csv`: weighted country-age-imputation cutoffs for both samples.
- `hfcs_build_audit.csv`: sample sizes and missingness by country, wave and imputation.
- `hfcs_country_wave_samples.csv`: compact country-wave summary using imputation 1.
- `hicp_annual_eurostat.csv`: cached annual all-items HICP indices, unless an external CPI file was supplied.

Country/wave differences remain. In particular, the additional Spanish `E1` vintage is retained but excluded pending confirmation of its reference year. Countries with bracket-only ages remain available for five-year age profiles, not single-age LOWESS. Proper multiple-imputation variance estimates and replicate-weight standard errors are handled at the estimation stage, not by treating the five imputations as independent observations.